**IMPORTS**

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DeiTFeatureExtractor, ViTModel
from PIL import Image
from tqdm import tqdm

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**CONFIG**

In [10]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "../checkpoints/early_fusion_proj/"
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

CODEBERT_CKPT = "../checkpoints/codebert_only/checkpoint-3760/"
VIT_CKPT = "../checkpoints/vit_only/deit_epoch_3.pt"

TEXT_TRAIN_DIR = "../text_files/train"
IMAGE_TRAIN_DIR = "../snapshots/train"
TEXT_VAL_DIR = "../text_files/valid"
IMAGE_VAL_DIR = "../snapshots/valid"

**LOAD MODELS**

In [3]:
from transformers import ViTForImageClassification, AutoImageProcessor

print("Loading fine-tuned CodeBERT...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
text_model = AutoModel.from_pretrained(CODEBERT_CKPT).to(DEVICE)
text_model.eval()

Loading fine-tuned CodeBERT...


Some weights of RobertaModel were not initialized from the model checkpoint at ../checkpoints/codebert_only/checkpoint-3760/ and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [4]:
print("Loading fine-tuned DeiT model...")
image_processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load your trained classification model first
trained_model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)

trained_model.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
trained_model.to(DEVICE)

# Extract just the ViT base model for embeddings
vit_model = trained_model.vit# This gives you the pure ViTModel
vit_model.to(DEVICE)
vit_model.eval()

Loading fine-tuned DeiT model...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

**DATASET**

In [5]:
class FusionDataset(Dataset):
    def __init__(self, text_dir, image_dir, tokenizer, image_processor):
        self.text_paths = []
        self.image_paths = []
        self.labels = []
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        # Scan text folders
        for label_folder in sorted(os.listdir(text_dir)):
            label_path = os.path.join(text_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            label = int(label_folder.split("_")[1])  # e.g., "Label_0" -> 0
            for txt_file in sorted(os.listdir(label_path)):
                if txt_file.endswith(".txt"):
                    self.text_paths.append(os.path.join(label_path, txt_file))
                    self.labels.append(label)

        # Scan image folders
        self.image_paths = []
        for label_folder in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            for img_file in sorted(os.listdir(label_path)):
                if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(label_path, img_file))

        # Ensure text_paths and image_paths are aligned
        assert len(self.text_paths) == len(self.image_paths), "Text and image counts must match!"

    def __len__(self):
        return len(self.text_paths)

    def __getitem__(self, idx):
        # ----- TEXT -----
        with open(self.text_paths[idx], "r") as f:
            text = f.read()
        encoding = self.tokenizer(
            text, return_tensors="pt", truncation=True, padding="max_length", max_length=512
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # ----- IMAGE -----
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image_tensor = self.image_processor(images=image, return_tensors="pt")
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, attention_mask, image_tensor, label


**DATA LOADING**

In [6]:
train_dataset = FusionDataset(TEXT_TRAIN_DIR, IMAGE_TRAIN_DIR, tokenizer, image_processor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = FusionDataset(TEXT_VAL_DIR, IMAGE_VAL_DIR, tokenizer, image_processor)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)


**MODEL ARCHITECTURE**

In [13]:
import torch
import torch.nn as nn

class CrossAttentionBlock(nn.Module):
    """One cross-attention block: text attends to image."""
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text_emb, image_emb):
        """
        text_emb: [B, L_t, D]
        image_emb: [B, L_i, D]
        """
        # Cross-attention: text queries attend to image keys/values
        attn_output, _ = self.cross_attn(query=text_emb, key=image_emb, value=image_emb)
        text_emb = text_emb + self.dropout(attn_output)
        text_emb = self.norm1(text_emb)

        # Feed-forward + residual + norm
        ffn_output = self.ffn(text_emb)
        text_emb = text_emb + self.dropout(ffn_output)
        text_emb = self.norm2(text_emb)

        return text_emb


class FusionClassifier(nn.Module):
    def __init__(self, text_model, vit_model, hidden_dim=512, num_classes=2, num_heads=8, num_layers=2):
        super().__init__()
        self.text_model = text_model
        self.vit_model = vit_model

        # Freeze backbone encoders
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.vit_model.parameters():
            p.requires_grad = False

        text_emb_dim = text_model.config.hidden_size
        vit_emb_dim = vit_model.config.hidden_size
        self.shared_dim = min(text_emb_dim, vit_emb_dim)

        # Project both modalities into same latent space
        self.text_proj = nn.Linear(text_emb_dim, self.shared_dim)
        self.image_proj = nn.Linear(vit_emb_dim, self.shared_dim)

        # Cross-Attention stack (can use >1 layer)
        self.cross_layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim=self.shared_dim, num_heads=num_heads)
            for _ in range(num_layers)
        ])

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.shared_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, image_tensor):
        # ---- 1️⃣ Encode Text ----
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = text_outputs.last_hidden_state  # [B, L_t, D_t]
        text_emb = self.text_proj(text_emb)        # [B, L_t, shared_dim]

        # ---- 2️⃣ Encode Image ----
        image_outputs = self.vit_model(**{k: v for k, v in image_tensor.items()})
        image_emb = image_outputs.last_hidden_state  # [B, L_i, D_i]
        image_emb = self.image_proj(image_emb)        # [B, L_i, shared_dim]

        # ---- 3️⃣ Cross-Attention Fusion ----
        for layer in self.cross_layers:
            text_emb = layer(text_emb, image_emb)

        # ---- 4️⃣ Use [CLS] token as fused representation ----
        fused_cls = text_emb[:, 0, :]  # [B, shared_dim]

        # ---- 5️⃣ Classify ----
        logits = self.classifier(fused_cls)
        return logits

**OPTIMIZERS && LOSS**

In [14]:
model = FusionClassifier(text_model, vit_model).to(DEVICE)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

**TRAINING LOOP && CHECKPOINTING**

In [15]:
from tqdm import tqdm
import torch
import os

# Assume:
# dataset = FusionDataset(TEXT_DIR, IMAGE_DIR, tokenizer, image_processor)
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

for epoch in range(EPOCHS):
    # -------------------- TRAIN --------------------
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]"):
        input_ids, attention_mask, image_tensor, labels = batch
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].to(DEVICE)

        logits = model(input_ids, attention_mask, image_tensor)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"🟩 Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f}")

    # -------------------- VALIDATION --------------------
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]"):
            input_ids, attention_mask, image_tensor, labels = batch
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(DEVICE)

            logits = model(input_ids, attention_mask, image_tensor)
            loss = criterion(logits, labels)
            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    print(f"🟦 Validation Loss: {avg_val_loss:.4f} | Accuracy: {val_acc*100:.2f}%")

    # -------------------- SAVE CHECKPOINT --------------------
    ckpt_path = os.path.join(OUTPUT_DIR, f"fusion_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✅ Checkpoint saved: {ckpt_path}")

Epoch 1/3 [Training]: 100%|██████████| 940/940 [01:45<00:00,  8.87it/s]


🟩 Epoch 1/3 | Train Loss: 0.1317


Epoch 1/3 [Validation]: 100%|██████████| 235/235 [00:22<00:00, 10.31it/s]


🟦 Validation Loss: 0.5742 | Accuracy: 82.34%
✅ Checkpoint saved: ../checkpoints/early_fusion_proj/fusion_epoch_1.pt


Epoch 2/3 [Training]: 100%|██████████| 940/940 [01:47<00:00,  8.74it/s]


...

🟦 Validation Loss: 0.6565 | Accuracy: 82.39%
✅ Checkpoint saved: ../checkpoints/early_fusion_proj/fusion_epoch_2.pt


Epoch 3/3 [Training]: 100%|██████████| 940/940 [01:48<00:00,  8.69it/s]


🟩 Epoch 3/3 | Train Loss: 0.1079


Epoch 3/3 [Validation]: 100%|██████████| 235/235 [00:23<00:00, 10.20it/s]


🟦 Validation Loss: 0.5839 | Accuracy: 82.34%
✅ Checkpoint saved: ../checkpoints/early_fusion_proj/fusion_epoch_3.pt


**TESTING**

In [1]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def test_on_dataset(text_dir, image_dir, tokenizer, image_processor, fusion_model, batch_size=4):
    """🧪 Test the fusion model on a single dataset"""

    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    fusion_model.eval()
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing Dataset 📝"):
            input_ids, attention_mask, image_tensor, labels = batch

            # Move text to device
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)

            # Move image tensors to device
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(DEVICE)

            # Forward pass
            logits = fusion_model(input_ids=input_ids, attention_mask=attention_mask, image_tensor=image_tensor)
            preds = torch.argmax(logits, dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    accuracy = total_correct / total_samples
    print(f"✅ Dataset Accuracy: {accuracy*100:.2f}% 🎉")
    return accuracy


In [2]:
FUSION_CKPT = "../checkpoints/early_fusion/fusion_epoch_3.pt"
fusion_model = FusionClassifier(text_model, vit_model)
fusion_model.to(DEVICE)
# Load checkpoint
state_dict = torch.load(FUSION_CKPT, map_location=DEVICE)
fusion_model.load_state_dict(state_dict)
fusion_model.eval()

NameError: name 'FusionClassifier' is not defined

In [ ]:
test_on_dataset(f"../text_files/valid/", f"../snapshots/valid", tokenizer, image_processor, fusion_model, batch_size=4)

Test_0


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.81it/s]


✅ Dataset Accuracy: 79.34% 🎉
Test_1


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.75it/s]


✅ Dataset Accuracy: 77.15% 🎉
Test_2


...

✅ Dataset Accuracy: 71.86% 🎉
Test_8


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.76it/s]


✅ Dataset Accuracy: 73.65% 🎉
Test_9


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.72it/s]

✅ Dataset Accuracy: 71.86% 🎉
